## Test Brute 

In [ ]:
import pandas as pd

def read_df(path):
    """
    Read a CSV file and return the DataFrame.
    """
    df = pd.read_csv(path)
    return df

def prepare_dataframe(df, series_id_value="France"):
    """
    Prepare the DataFrame:
    - Create time_idx
    - Add series_id if missing
    - Keep only useful columns
    """
    df = df.copy()

    # Create time_idx
    df["time_idx"] = range(len(df))

    # Add series_id if not present
    if "series_id" not in df.columns:
        df["series_id"] = series_id_value

    return df[["time_idx", "series_id", "Consommation"]]

def split_train_test(df, test_year=2022, datetime_col="Datetime"):
    """
    Split the prepared DataFrame into train and test sets based on the year of the datetime column.
    """
    if datetime_col not in df.columns:
        raise ValueError(f"Datetime column '{datetime_col}' not found in DataFrame.")

    df[datetime_col] = pd.to_datetime(df[datetime_col])  # Just to be sure
    df = df.sort_values(datetime_col).reset_index(drop=True)
    df = df.set_index(datetime_col)

    df_train = df[df.index.year < test_year].reset_index()
    df_test = df[df.index.year == test_year].reset_index()

    return df_train, df_test

In [ ]:
df = read_df(path = "cleaned_data/cleaned_without_nan_non_aggregate_data_15_22.csv")
df_train, df_test = split_train_test(df, test_year=2022)
df_train = prepare_dataframe(df_train, series_id_value="France")

## Test avec IC + Validation + autre métrique

In [ ]:
# Imports nécessaires
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE, RMSE
from torch.utils.data import DataLoader
from lightning.pytorch import Trainer
import torch
from lightning.pytorch.loggers import TensorBoardLogger
torch.set_float32_matmul_precision('medium')

# Logger
logger = TensorBoardLogger("lightning_logs", name="tft_model")

# ---------------------
# Paramètres de fenêtre
max_encoder_length = 48
max_prediction_length = 48
batch_size = 64

# ---------------------
# Préparation des jeux training/validation
training_cutoff = df_train["time_idx"].max() - 48 * 365  # réserver 1 an pour test (~365 jours)

# Jeu d'entraînement
training = TimeSeriesDataSet(
    df_train[df_train.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="Consommation",
    group_ids=["series_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    time_varying_known_reals=["time_idx"],
    time_varying_unknown_reals=["Consommation"],
    target_normalizer=None,
    add_relative_time_idx=True,
    add_target_scales=True, 
    add_encoder_length=True
)

# Jeu de validation
validation = TimeSeriesDataSet.from_dataset(
    training, df_train, predict=True, stop_randomization=True
)

# ---------------------
# Créer les DataLoaders
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=16)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size * 10, num_workers=16)

# ---------------------
# Définir le modèle TFT
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.01,
    hidden_size=512,
    attention_head_size=16,
    dropout=0.1,
    hidden_continuous_size=124,
    output_size=7,  # 7 quantiles par défaut
    loss=QuantileLoss(),
)

# ---------------------
# Entraînement
trainer = Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices=1,
    gradient_clip_val=0.1,
    logger=logger,
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

In [ ]:
# ---------------------
# Fichier de logs TensorBoard
writer = SummaryWriter(log_dir="lightning_logs/tft_model")

# ---------------------
# Prédictions sur la validation
raw_predictions = tft.predict(
    val_dataloader,
    mode="raw",
    return_x=True
)

# ---------------------
# Sélectionner la médiane (quantile 0.5) pour l'évaluation
pred_median = raw_predictions.output.prediction[:, :, 3]  # le 4ᵉ quantile, index 3

# ---------------------
# Calcul des métriques (MAE, RMSE)
true_values = raw_predictions.x["decoder_target"]

mae = MAE()(pred_median, true_values)
rmse = RMSE()(pred_median, true_values)
r2 = r2_score(true_values.detach().cpu().numpy().flatten(), pred_median.detach().cpu().numpy().flatten())

print("MAE:", mae.item())
print("RMSE:", rmse.item())

# ---------------------
# Calcul du Coverage (entre quantile 5% et 95%)
pred_5 = raw_predictions.output.prediction[:, :, 0]  # 5e percentile
pred_95 = raw_predictions.output.prediction[:, :, 6]  # 95e percentile

is_covered = ((true_values >= pred_5) & (true_values <= pred_95)).float()
coverage = is_covered.mean()

print(f"Coverage (entre 5% et 95%) : {coverage.item() * 100:.2f}%")
# ---------------------
# Log des métriques dans TensorBoard
writer.add_scalar("Validation/MAE", mae.item(), 0)
writer.add_scalar("Validation/RMSE", rmse.item(), 0)
writer.add_scalar("Validation/R2", r2, 0)

# ---------------------
# Fermer le writer proprement
writer.close()

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from pytorch_forecasting import TimeSeriesDataSet
from torch.utils.data import DataLoader

def build_prediction_dataset(df, encoder_length=48, prediction_length=48):
    dataset = TimeSeriesDataSet(
        df,
        time_idx="time_idx",
        target="Consommation",
        group_ids=["series_id"],
        max_encoder_length=encoder_length,
        max_prediction_length=prediction_length,
        time_varying_known_reals=["time_idx"],
        time_varying_unknown_reals=["Consommation"],
        target_normalizer=None,
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True
    )
    return dataset

def create_dataloader(dataset, batch_size=1):
    return dataset.to_dataloader(train=False, batch_size=batch_size, num_workers=0)

def predict_from_dataloader(model, dataloader):
    raw_predictions = model.predict(dataloader, mode="raw", return_x=True)
    return raw_predictions

def plot_predictions_from_dataframe(df_test, model, n=2, seed=None):
    if seed is not None:
        np.random.seed(seed)
    
    df_test = df_test.copy()
    df_test["Datetime"] = pd.to_datetime(df_test["Datetime"])
    df_test = df_test.sort_values("Datetime").reset_index(drop=True)

    available_days = df_test["Datetime"].dt.floor('d').unique()
    selected_days = np.random.choice(available_days[:-1], size=n, replace=False)

    for day in selected_days:
        input_day = df_test[(df_test["Datetime"] >= day) & (df_test["Datetime"] < day + pd.Timedelta(days=1))]
        next_day = df_test[(df_test["Datetime"] >= day + pd.Timedelta(days=1)) & (df_test["Datetime"] < day + pd.Timedelta(days=2))]

        if len(input_day) < 48 or len(next_day) < 48:
            print(f"Skipping day {pd.Timestamp(day).date()} because not enough data.")
            continue

        small_df = pd.concat([input_day, next_day], ignore_index=True).copy()
        small_df["time_idx"] = range(len(small_df))
        if "series_id" not in small_df.columns:
            small_df["series_id"] = "France"
        
        small_dataset = build_prediction_dataset(small_df)
        small_dataloader = create_dataloader(small_dataset)
        raw_preds = predict_from_dataloader(model, small_dataloader)

        preds = raw_preds.output.prediction
        x = raw_preds.x
        true_values = x["decoder_target"][0]

        pred_5 = preds[0, :, 0]
        pred_50 = preds[0, :, 3]
        pred_95 = preds[0, :, 6]

        # Plot
        plt.figure(figsize=(12, 6))
        plt.plot(true_values.cpu(), label="Consommation réelle", color="black")
        plt.plot(pred_50.cpu(), label="Prédiction médiane", color="blue")
        plt.fill_between(
            np.arange(pred_5.shape[0]),
            pred_5.cpu(),
            pred_95.cpu(),
            color="blue",
            alpha=0.3,
            label="Intervalle de confiance 90%"
        )

        max_pred = pred_95.max().item()
        min_pred = pred_5.min().item()
        plt.hlines(max_pred, 0, 47, colors="red", linestyles="dashed", label=f"Max préd {max_pred:.1f}")
        plt.hlines(min_pred, 0, 47, colors="green", linestyles="dashed", label=f"Min préd {min_pred:.1f}")

        plt.title(f"Prédiction de la consommation pour le {pd.Timestamp(day).date()}")
        plt.xlabel("Pas de temps (30min)")
        plt.ylabel("Consommation (MW)")
        plt.legend()
        plt.grid()
        plt.show()

In [ ]:
plot_predictions_from_dataframe(df_test, tft, n=2, seed=42)